# Implementation of BlackJack Policy Evaluation With Dual-Q learning

Possible action for every state :
* Hit (you pick a card and the game goes on)
* Stay (the dealer reveals their face down card and picks card until their sum is more than 17). This will lead to a terminal state (you win / loose)


In [1]:

from enum import Enum
from itertools import product
import random


class Action(Enum):
    STICK = 0
    HIT = 1


def draw_card():
    # Infinite deck: A=1, J/Q/K=10
    return min(random.randint(1, 13), 10)


class Hand:
    def __init__(self):
        self.cards = []

    def add_card(self, card):
        self.cards.append(card)

    def value(self):
        total = sum(self.cards)

        # Count one Ace as 11 if possible
        if 1 in self.cards and total + 10 <= 21:
            return total + 10

        return total

    def is_bust(self):
        return self.value() > 21


def policy(state):
    # Naive fixed policy
    return Action.STICK if state >= 20 else Action.HIT


def dealer_policy(hand):
    return Action.HIT if hand.value() < 17 else Action.STICK


def play_dealer(dealer):
    # Dealer plays after the player sticks
    while dealer_policy(dealer) == Action.HIT:
        dealer.add_card(draw_card())


def outcome(player, dealer):
    # Determine the terminal reward
    if player.is_bust():
        return -1

    if dealer.is_bust() or player.value() > dealer.value():
        return 1

    if player.value() == dealer.value():
        return 0

    return -1


def update_Q(Q_update, Q_other, reward,
             state, action, next_state,
             done=False, gamma=1.0, alpha=0.1):

    if done:
        target = reward
    else:
        # Select the maximizing action using Q_update
        a_max = max(
            [Action.STICK, Action.HIT],
            key=lambda a: Q_update[(next_state, a)]
        )

        # Evaluate that action using the other Q-table
        target = reward + gamma * Q_other[(next_state, a_max)]

    # TD update
    Q_update[(state, action)] += alpha * (
        target - Q_update[(state, action)]
    )


def policy_evaluation(num_episodes=100_000,
                      epsilon=0.1,
                      gamma=1.0,
                      alpha=0.1,
                      use_fixed_policy=False):

    actions = [Action.STICK, Action.HIT]

    # Initialize both Q-tables independently
    state_action_pairs = list(product(range(4, 22), actions))

    Q1 = {
        pair: random.uniform(-0.01, 0.01)
        for pair in state_action_pairs
    }

    Q2 = {
        pair: random.uniform(-0.01, 0.01)
        for pair in state_action_pairs
    }

    # Each iteration is a complete Blackjack episode
    for episode in range(num_episodes):

        player = Hand()
        dealer = Hand()

        # Deal two cards to each
        for _ in range(2):
            player.add_card(draw_card())
            dealer.add_card(draw_card())

        state = player.value()

        while True:

            # Select action using the behavior policy
            if use_fixed_policy:
                action = policy(state)

            elif random.random() < epsilon:
                # Exploration
                action = random.choice(actions)

            else:
                # Exploitation using Q1 + Q2
                action = max(
                    actions,
                    key=lambda a: (
                        Q1[(state, a)] + Q2[(state, a)]
                    )
                )

            # Execute action in the environment
            if action == Action.HIT:

                player.add_card(draw_card())
                next_state = player.value()

                done = player.is_bust()
                reward = -1 if done else 0

            else:
                # Player sticks: dealer finishes the game
                play_dealer(dealer)

                reward = outcome(player, dealer)
                next_state = None
                done = True

            # Double Q-learning: update one table randomly
            if random.random() < 0.5:
                update_Q(
                    Q1, Q2, reward,
                    state, action, next_state,
                    done, gamma, alpha
                )
            else:
                update_Q(
                    Q2, Q1, reward,
                    state, action, next_state,
                    done, gamma, alpha
                )

            if done:
                break

            state = next_state

    return Q1, Q2


# Train the agent
Q1, Q2 = policy_evaluation(num_episodes=100_000)

# Extract the learned greedy policy
for state in range(4, 22):
    best_action = max(
        Action,
        key=lambda a: Q1[(state, a)] + Q2[(state, a)]
    )

    print(f"Player sum {state}: {best_action.name}")


Player sum 4: HIT
Player sum 5: HIT
Player sum 6: HIT
Player sum 7: HIT
Player sum 8: HIT
Player sum 9: HIT
Player sum 10: HIT
Player sum 11: HIT
Player sum 12: HIT
Player sum 13: HIT
Player sum 14: HIT
Player sum 15: HIT
Player sum 16: HIT
Player sum 17: STICK
Player sum 18: STICK
Player sum 19: STICK
Player sum 20: STICK
Player sum 21: STICK


The policy is made greedy w.r.t Q function for a given $S_t$ as follows

$$a^* = \arg \max_a(Q_1(S_t,a)+Q_2(S_t,a))$$

Which is similar to maximizing the average of the two Q functions